# 08 · Agentic RAG: retrieval becomes a tool
**Concept:** an agent decides *whether*, *where* and *how often* to retrieve, then answers with citations.
**TAC scenario:** an L2 engineer following the evidence across the KB, release notes, the case archive and product guides.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from typing import Optional
from langchain.tools import tool
from rag_common import format_docs, chroma_filter

def search(query: str, doc_type: str, product: Optional[str] = None, k: int = 4) -> str:
    flt = chroma_filter(product=product, doc_type=doc_type)
    docs = vectorstore.similarity_search(query, k=k, filter=flt)
    return format_docs(docs) or "No results."

@tool
def kb_search(query: str, product: Optional[str] = None) -> str:
    """Search knowledge-base articles for symptoms, root causes and workarounds.
    Optionally restrict to a product name such as 'Nimbus NF-300'."""
    return search(query, "kb", product)

@tool
def release_notes_search(query: str, product: Optional[str] = None) -> str:
    """Search release notes to find which software release fixes an issue or what a release contains."""
    return search(query, "release_note", product)

@tool
def ticket_search(query: str, product: Optional[str] = None) -> str:
    """Search resolved customer tickets to see whether an issue was seen before and how it was resolved."""
    return search(query, "ticket", product, k=5)

@tool
def guide_search(query: str, product: Optional[str] = None) -> str:
    """Search product configuration guides for how-to steps and CLI commands."""
    return search(query, "guide", product)

TOOLS = [kb_search, release_notes_search, ticket_search, guide_search]

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(
    model,
    tools=TOOLS,
    system_prompt=("You are a TAC L2 engineer. Use the search tools to gather evidence before answering. "
                   "Cite every claim as [doc_id]. If the evidence is insufficient, say INSUFFICIENT_DATA."),
    middleware=[ModelCallLimitMiddleware(run_limit=8, exit_behavior="end")],     # cap the search loop
)

def ask(q: str):
    print(f"❓ {q}")
    for update in agent.stream({"messages": [{"role": "user", "content": q}]}, stream_mode="updates"):
        for node, data in update.items():
            if not data or "messages" not in data:
                continue
            msg = data["messages"][-1]
            for tc in getattr(msg, "tool_calls", None) or []:
                print(f"   🔎 {tc['name']}({tc['args']})")
            if msg.type == "ai" and not getattr(msg, "tool_calls", None):
                print(f"\n💬 {msg.content}\n")

## 1. A simple lookup: one search is enough

In [ ]:
ask("What causes ERR-TUN-2207 on Atlas SD-WAN?")

## 2. A multi-hop investigation: the agent chains sources

In [ ]:
ask("A Kestrel Logistics firewall on Nimbus 5.1.3 drops tunnels every night. Is it a known issue, which release fixes it, "
    "and have other customers fixed it with a workaround?")

## 3. Something the knowledge base can't answer

In [ ]:
ask("Does the Nimbus NF-300 integrate with the Zephyr SIEM platform?")

---
**🔧 TAC takeaway:** Agentic RAG is an L2 engineer choosing between the KB, release notes and the case archive, following the evidence instead of running one fixed search. The model-call limit keeps the investigation bounded.